# High-order element-by-element curl recovery

**Local vector-potential recovery works at p=2 and p=3 on the compatible discrete fields tested here.** This notebook saves a prototype and its numerical evidence. It does not replace the global A-T CLN calculation.

Biro and Koster, *Generating a Cauer Ladder Network Representation of Eddy Current Fields Using Scalar Potentials*, [Compumag 2021 author paper](https://www.compumag.org/Proceedings/2021_Cancun/www.conftool.pro/compumag2021/index.php/OA1-2-299ecbe.pdf?filename=OA1-2-299.pdf&form_id=299&form_version=final&ismobile=false&page=downloadPaper), Section IV, recover vector potentials with element-local minimum-residual systems while retaining degrees of freedom fixed by earlier elements. The electric and magnetic scalar corrections remain global Poisson problems. The paper refers to Biro and Gyimesi, IEEE Transactions on Magnetics 57(6), 6301704, [DOI 10.1109/TMAG.2021.3063851](https://doi.org/10.1109/TMAG.2021.3063851).

The consulted primary paper does not assert a p=1-only limitation and its example describes 36-edge hexahedral elements. The precise source of a p=1-only theorem has not been identified. The extension below is our independently tested high-order construction, not an attributed theorem.

**Follow-up on actual CLN meshes:** [Notebook 19](19_element_cln_integration.ipynb) shows that this greedy one-pass sweep fails on the curved wire. A compatible global trace solve with element-local recovery reproduces the full high-order A-T CLN. The affine-cube result below must not be generalized to arbitrary meshes.

## Local construction and compatibility

For an NGSolve HCurl element, form the unperturbed matrix

$$K_e=\int_e \operatorname{curl}N_i\cdot\operatorname{curl}N_j\,dV=C_e^T C_e.$$

Positive eigenmodes define the local factor $C_e$; null gradient modes remain unpenalized. Given a globally compatible discrete target, its element data are $b_e=C_e a_*$. Keep shared edge and face degrees of freedom fixed once visited, and solve the remaining local least-squares system. High order adds face and interior modes; assuming only edge unknowns would discard them. NGSolve supplies the basis, orientation and element mapping.

A separate matrix-free LSMR calculation applies all local factors while sharing global trace unknowns. An independently assembled global curl-curl pseudoinverse supplies another reference. We compare curls, since potentials can differ by gradients. A numerical integrated curl norm checks the physical field separately from the local algebra.

This experiment uses a simply connected affine cube with 12 tetrahedra, full HCurl spaces, p=1/2/3, two deterministic random compatible target coefficient vectors per order, and forward/reverse/shuffled element sweeps. It tests recovery of the complete discrete curl space, not continuum approximation or a manufactured analytic field. No mass perturbation or PARDISO is used.

In [1]:
from pathlib import Path
import json
root=Path.cwd()
if not (root/'validation').is_dir(): root=root.parent
data=json.loads((root/'docs/data/element_curl_recovery.json').read_text())
print('host:',data['host'],'NGSolve:',data['ngsolve'],'threads:',data['threads'])
print('p  seed  DOFs  local rank  worst single sweep  global reference difference')
for c in data['cases']:
    print(f"{c['order']}  {c['seed']:4}  {c['ndof']:4}  {c['local_curl_ranks'][0]:10}  {max(c['greedy_curl_errors'].values()):18.3e}  {c['iterative_reference_curl_difference']:.3e}")
print('largest integrated curl error:',max(c['integrated_curl_error'] for c in data['cases']))

host: intel11 NGSolve: 6.2.2607 threads: 4
p  seed  DOFs  local rank  worst single sweep  global reference difference
1    17    52           3           5.674e-16  1.842e-14
1    29    52           3           6.299e-16  1.291e-14
2    17   168          11           1.053e-15  1.456e-12
2    29   168          11           1.576e-15  1.781e-12
3    17   392          26           8.635e-14  6.280e-12
3    29   392          26           1.788e-13  6.879e-12
largest integrated curl error: 6.865573869798364e-12


## Incompatible data must fail a recovery gate

Independent random element curl coordinates generally violate shared-trace compatibility. Their best compatible reconstruction leaves a large residual. Silently accepting a local least-squares answer would change the prescribed field. This control checks that the residual reveals the inconsistency.

In [2]:
for c in data['cases']:
    print('p=',c['order'],'seed=',c['seed'],'incompatible residual=',round(c['incompatible_curl_error'],4))

p= 1 seed= 17 incompatible residual= 0.6736
p= 1 seed= 29 incompatible residual= 0.7783
p= 2 seed= 17 incompatible residual= 0.68
p= 2 seed= 29 incompatible residual= 0.6443
p= 3 seed= 17 incompatible residual= 0.5835
p= 3 seed= 29 incompatible residual= 0.5448


## Result and boundary of the result

All 18 single sweeps pass: p=3 maximum relative curl error is below 1.8e-13. Element-operator LSMR and the independent global reference agree within 7e-12. Incompatible data leave residuals above 54%. These are local recovery results, not a proof that every element traversal works on every mesh.

The next integration gate is to take actual CLN source fields, establish their discrete curl-image compatibility, reconstruct A/T locally, apply the global scalar corrections, and compare coefficients, energy and original equations against the existing A-T route. Curved meshes, topology and material jumps need explicit gates. No speed claim follows from the prototype; the global pseudoinverse and LSMR are validation references.

[A-T accuracy conclusion](17_formulation_ranking.ipynb), [native producer](../validation/element_curl_recovery.py), [saved native evidence](data/element_curl_recovery.json), [portable evidence gate](../validation/validate_element_curl_recovery.py).

Reproduce on an NGSolve machine:

```text
python validation/element_curl_recovery.py --output docs/data/element_curl_recovery.json
python validation/validate_element_curl_recovery.py
```